# RAG talent search with real embedding models and an LLM (Colab)
> Needs internet. It could not be executed where the project was written (no hub / API access), so the commands mirror the tested CLI.
> Use a **local** LLM (Ollama) for real candidate data; a cloud LLM receives redacted resume excerpts.

In [ ]:
!git clone https://github.com/salikhussain71-code/rag-talent-search-engine.git
%cd rag-talent-search-engine
!pip install -q -e ".[all]"
!python -m talent_rag info

## 1. Get the Kaggle dataset
Download *Resume Entities for NER* from Kaggle (upload the file here, or use the Kaggle API) and put it in `data/raw/`.

In [ ]:
from google.colab import files
uploaded = files.upload()          # upload "Entity Recognition in Resumes.json"
import shutil, os
os.makedirs("data/raw", exist_ok=True)
for name in uploaded: shutil.move(name, f"data/raw/{name}")
!python -m talent_rag stats --data "data/raw/Entity Recognition in Resumes.json"

## 2. Build the index with a real sentence-embedding model (downloads the model once)

In [ ]:
!python -m talent_rag build-index --data "data/raw/Entity Recognition in Resumes.json" --out outputs/kaggle_index --embedder bge-small --store faiss

## 3. Search + LLM explanation (Gemini or OpenAI key from Colab secrets)

In [ ]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
!python -m talent_rag search "Find me a Junior Data Analyst who knows SQL and Tableau" --index outputs/kaggle_index --explain gemini:gemini-2.5-flash

## 4. Compare embedders and check for bias on the real data

In [ ]:
!python -m talent_rag evaluate --data "data/raw/Entity Recognition in Resumes.json" --embedders lsa minilm bge-small --compare-anonymization --out outputs/kaggle_evaluation
!python -m talent_rag bias-audit --data "data/raw/Entity Recognition in Resumes.json" --embedder bge-small --out outputs/kaggle_bias

## 5. Chat UI
Run `streamlit run app/streamlit_app.py` on your own machine (Colab needs a tunnel), pointing `TALENT_INDEX` at the index you built.